In [11]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.1",
    temperature=0,
    # num_threads=29,
    num_gpu=1
 )

In [123]:

from typing import Optional
from pydantic import Field
from pydantic._internal._model_construction import ModelMetaclass                                                                                                                                                                        
from langchain_core.prompts import SystemMessagePromptTemplate                                                                                                                                                                
from langchain_core.messages import SystemMessage, HumanMessage

from langchain.output_parsers import PydanticOutputParser


class AgenticSystemPrompt:                                                                                                                                                                                                    
    """ System Prompts for all the use cases                                                                                                                                                                                  
    """                                                                                                                                                                                                                       
    def __init__(self):                                                                                                                                                                                                       
        """ Initiator of agentic system prompt """                                                                                                                                                                            
        # self.config = config                                                                                                                                                                                                  
        # self.logging = self.config.logging                                                                                                                                                                                    
        
    @staticmethod                                                                                                                                                                                                                          
    def get_docstring(pydantic_object: Optional[ModelMetaclass] = None) -> SystemMessage:
        """ Get the predefined prompt on validating the response """                                                                                                                                                          
        prompt = """As a highly skilled prompt engineer specializing in Python documentation, your task is to generate a comprehensive Google-style docstring for a Python function or method.

        I will provide you with the following information:
        
        1.  **Function/Method Arguments:** A clear description of each argument, including its name and expected data type.
        2.  **Return Value(s):** A description of what the function/method returns, including the data type(s).
        3.  **(Optional) Side Effects:** Any observable effects beyond the return value (e.g., modifying global variables, writing to a file).
        4.  **(Optional) Raises:** Any exceptions that the function/method might raise, along with the conditions under which they are raised.
        5.  **(Optional) Notes:** Any additional important information or considerations.
        6.  **(Optional) Examples:** Illustrative examples of how to use the function/method.
        7.  **(Optional) Current Function/Method Name:** The existing name of the function or method.
            
        Based on this input, you will generate a well-formatted Google-style docstring. Additionally, you will suggest a more effective and descriptive name for the function or method if the provided name is unclear or not aligned with its functionality.
        """
        prompt = """
        As a highly skilled python developer with 20 years of expierence, your task is to take a python function arguments 
        and output returned from the model. With those information you will have to create a **google docstring**.

        Instructions:
        1. You should give the response in the format given.
        2. Don't halucinate if you dont know tell us that you don't know.
        3. You store the **doctstring**, **alternative_fun_name** in python string and fit the structure
        4. Do not create any preamble, just give the structured results.

        For example,

        Function input:
        users=List[User]
        
        Function output:
        list[str]

        description: Email retireved from the email attribute

        Model Response
        docstring: "Iterates through a list of user objects and retrieves the value
        of the 'email' attribute for each user. Returns a new list
        containing these email addresses.
        
        Args:
            users (list[User]): A list of user objects, where each object
                is expected to have an 'email' attribute (str).
        
        Returns:
            list[str]: A list of strings, where each string is an email
                address extracted from the input user objects.
        
        Examples:
            >>> users = [{'name': 'Alice', 'email': 'alice@example.com'}, {'name': 'Bob', 'email': 'bob@example.org'}]
            >>> extract_user_emails(users)
            ['alice@example.com', 'bob@example.org']"
            
        alternative_fun_name:  "extract_user_emails"
        """
        prompt = """
        You are given the function signature and its description. Your task is to generate a
        detailed Python docstring and suggest an alternative function name that clearly describes the functionality.

        Instructions:
        1. You should give the response in the format given.
        2. Don't halucinate if you dont know tell us that you don't know.
        3. You store the **docstring**, **alternative_fun_name** in python string and fit the structure
        4. Do not create any preamble, just give the structured results.
        5. The docstring should be in google docs format

        ---------------Example on the usage ------------------------------------
        Function input:
        ```python
        users: List[User]
        ```
        Function output:
        ```python
        list[str]
        ```
        Description:
        Email retrieved from the email attribute of each user.

        Model Response:
        "
        Iterates through a list of user objects and retrieves the value
        of the 'email' attribute for each user. Returns a new list
        containing these email addresses.
        
        Args:
        users (list[User]): A list of user objects, where each object
        is expected to have an 'email' attribute (str).
        
        Returns:
        list[str]: A list of strings, where each string is an email
        address extracted from the input user objects.
        
        Examples:
        >>> users = [{'name': 'Alice', 'email': 'alice@example.com'}, {'name': 'Bob', 'email': 'bob@example.org'}]
        >>> extract_user_emails(users)
        ['alice@example.com', 'bob@example.org']
        "
        alternative_fun_name: "extract_user_emails"
        ------------------------------------------------------------------------------------
        """
        prompt = """
        You are given the function signature and its description. Your task is to generate a
        detailed Python docstring and suggest an alternative function name that clearly describes the functionality.
        
        Instructions:
        1. You should give the response in the format given.
        2. Don't hallucinate. If you don't know, say so clearly.
        3. Store the **docstring** and **alternative_fun_name** in Python strings as per the format.
        4. Do not create any preamble or explanations, just return the structured result.
        5. The docstring should follow Google-style Python docstring format.
        """
        
   
        if pydantic_object:
            parser = PydanticOutputParser(pydantic_object=pydantic_object)
            instructions = parser.get_format_instructions()
            
            prompt += f"Instruction on the structured output: {instructions}"
            
            sys_prompt = SystemMessage(prompt)
            return parser, sys_prompt
        
        sys_prompt = SystemMessage(prompt)
        return None, sys_prompt
            

        

In [151]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="gemma3",
    temperature=50,
    num_threads=29,
    num_gpu=1
 )

In [152]:
from datetime import datetime
from typing import Tuple
from pydantic import BaseModel

start_time = datetime.now()


class DocstringStructure(BaseModel):
    docstring: str = Field(...,
                           description="A Google docstring of the python function")
    alternative_fun_name: str = Field(...,
                                           description="A suggested function name for the respective description")
    

def create_docstring(function_in: str,
                     function_out: str,
                     description: Optional[str] = None) -> Tuple[str, str]:
    """
    Generates a Google-style docstring based on the provided function signature and description.
    
    Args:
        function_in (str): A string representing the input arguments of the function,
            including their types (e.g., "users=List[User], threshold=int").
        function_out (str): A string representing the return type of the function
            (e.g., "list[str]", "bool", "Tuple[int, str]").
        description (Optional[str], optional): A brief description of what the
            function does. Defaults to None.
    
    Returns:
        Tuple[str, str]: A tuple containing the generated Google-style docstring
            and a suggested alternative function name based on the description.
            If no description is provided, the alternative function name will be
            an empty string.
    
    Examples:
        >>> create_docstring("users=List[User]", "list[str]", "Extracts email addresses from a list of users.")
        ('Extracts email addresses from a list of users.\\n\\nArgs:\\n    users (list[User]): A list of user objects.\\n\\nReturns:\\n    list[str]: A list of extracted email addresses.', 'extract_user_emails')
    
        >>> create_docstring("data=pd.DataFrame, column_name=str", "pd.Series")
        ('Args:\\n    data (pd.DataFrame): The input DataFrame.\\n    column_name (str): The name of the column to select.\\n\\nReturns:\\n    pd.Series: The selected column as a Series.', 'select_column')
    
        >>> create_docstring("x=int, y=int", "int")
        ('Args:\\n    x (int): The first integer.\\n    y (int): The second integer.\\n\\nReturns:\\n    int: The result of the operation.', '')
    """
    prompt = f"""

    Function Input:
    {function_in}    

    Function Output:
    {function_out}
    """

    if description:
        prompt += f"\nDescription: {description}"

    
    docstring_usr_prompt = HumanMessage(prompt)
    
    response = llm.invoke([sys_msg_docstring, docstring_usr_prompt])
    print(response)
    # formated_response = parser.invoke(response)
    # print(type(formated_response))
    # print(formated_response)
    return response



In [153]:
function_in = """function_in: str,
                 function_out: str,
                 description: Optional[str]"""
function_out = "Tuple[str, str] "
description = """A function to create a docstring with given the function_in (function input)
             function_out (function output) and description of the function usecases 
             """ 

In [154]:
# prompt = f"""
# Instructions

# Function Input:
# {function_in}    

# Function Output:
# {function_out}
# """

# if description:
#     prompt += f"\nDescription: {description}"

    
# docstring_usr_prompt = HumanMessage(prompt)
# print(docstring_usr_prompt)
# docstring_usr_prompt.content

In [156]:
response = create_docstring(function_in,
                            function_out,
                            description)

content='def stringcat\n{ doc: \'\'\'doc generation\'\'\' ) #returns one, generates given desc that functions, takes parameter doc \'\'\'\', } func :\nstring\ntocat generate.description doc input togen description\ngenerate returns desc\n\nstr docfunctionfunctioncat\' returns func\n\n## Here `returns `cat inputcat generate `\'\'\'Doc\\" \\\\str.description , desc .   A generator that create one generation."""     function\n\ncat,doc\'  ""Returns `tostrc\n\ncatfunc`output    """ `functions        is an,    generator for return""" "" """ generates generator generates one string based cat\nis also called "returns and creates return returns tostr, or\n\ngenerator\nif output outputgenerator funcgenerate string function doc\n\ndescription returns description   descfunction_funcout\'and stringdescfuncifoutput  the generationgen    cat returns\n""is as a toReturn ifand ,or\n# Returns Tuple\\ <h1>""" # return\' <b>to str\n<stringreturnsand """anddescription ,generateifdescanfor\n\ndoc 9andoutpu

In [159]:
parser, sys_msg_docstring = AgenticSystemPrompt.get_docstring(DocstringStructure)    

In [160]:
parser.invoke(response)

OutputParserException: Invalid json output: def stringcat
{ doc: '''doc generation''' ) #returns one, generates given desc that functions, takes parameter doc '''', } func :
string
tocat generate.description doc input togen description
generate returns desc

str docfunctionfunctioncat' returns func

## Here `returns `cat inputcat generate `'''Doc\" \\str.description , desc .   A generator that create one generation."""     function

cat,doc'  ""Returns `tostrc

catfunc`output    """ `functions        is an,    generator for return""" "" """ generates generator generates one string based cat
is also called "returns and creates return returns tostr, or

generator
if output outputgenerator funcgenerate string function doc

description returns description   descfunction_funcout'and stringdescfuncifoutput  the generationgen    cat returns
""is as a toReturn ifand ,or
# Returns Tuple\ <h1>""" # return' <b>to str
<stringreturnsand """anddescription ,generateifdescanfor

doc 9andoutput: and#  